<a href="https://colab.research.google.com/github/vaibhavchavhan45/langchain-core-components/blob/main/langchain_retrievers_practise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

In [ ]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia langchain-google-genai

# Wikipedia Retriver

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

In [ ]:
# Initialise a retriever (fields are optional)
retriever = WikipediaRetriever(top_k_results = 2, lang = "en")

In [ ]:
# define user query
query = "The geopolitical history of Russia and Ukraine from the perspective of USA"

In [ ]:
# get relevant documents from wikipedia
docs = retriever.invoke(query)

In [ ]:
# print those 2 documents ----> [Document(metadata = .... , page_content = ..... )]
docs

[Document(metadata={'title': 'Russo-Ukrainian war', 'summary': 'The Russo-Ukrainian war began in February 2014 and is ongoing. Following Ukraine\'s Revolution of Dignity, Russia occupied Crimea and annexed it from Ukraine. It then supported Russian-backed armed groups who started a war in the eastern Donbas region against Ukraine\'s military. In 2018, Ukraine declared the region to be occupied by Russia. The first eight years of conflict also involved naval incidents and cyberwarfare. In February 2022, Russia launched a full-scale invasion of Ukraine and began occupying more of the country, starting the current phase of the war, the biggest conflict in Europe since World War II. The war has resulted in a refugee crisis and hundreds of thousands of deaths.\nIn early 2014, the Euromaidan protests led to the Revolution of Dignity and the ousting of Ukraine\'s pro-Russian president Viktor Yanukovych. Shortly after, pro-Russian protests began in parts of southeastern Ukraine, while unmarked

In [ ]:
for i, doc in enumerate(docs):
  print(f"\n --Result {i+1} --")
  print(f"\n Content : \n {doc.page_content}")

# OR
# for doc in docs:
#   print(doc.page_content)



 --Result 1 --

 Content : 
 The Russo-Ukrainian war began in February 2014 and is ongoing. Following Ukraine's Revolution of Dignity, Russia occupied Crimea and annexed it from Ukraine. It then supported Russian-backed armed groups who started a war in the eastern Donbas region against Ukraine's military. In 2018, Ukraine declared the region to be occupied by Russia. The first eight years of conflict also involved naval incidents and cyberwarfare. In February 2022, Russia launched a full-scale invasion of Ukraine and began occupying more of the country, starting the current phase of the war, the biggest conflict in Europe since World War II. The war has resulted in a refugee crisis and hundreds of thousands of deaths.
In early 2014, the Euromaidan protests led to the Revolution of Dignity and the ousting of Ukraine's pro-Russian president Viktor Yanukovych. Shortly after, pro-Russian protests began in parts of southeastern Ukraine, while unmarked Russian troops occupied Crimea. Russi

# Vector Store Retriever

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [ ]:
# source documents
docs = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="LangGraph helps developers to build Agentic system"),
    Document(page_content="FAISS is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [ ]:
# Embedding model
embedding_model = GoogleGenerativeAIEmbeddings(
    model = 'models/text-embedding-004'
)

In [ ]:
# vector store
vectorstores = Chroma.from_documents(
    documents = docs,
    embedding = embedding_model,
    collection_name = "vector-store-retriever-collection"
)

In [ ]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstores.as_retriever(search_kwargs = {"k" : 2})

In [ ]:
query = "What is Langchain used for?"

In [ ]:
result = retriever.invoke(query) # preferred
# OR
# results = vectorstore.similarity_search(query, k=2)

In [ ]:
for i, doc in enumerate(result):
  print(f"\n -- Result {i+1} -- \n")
  print(doc.page_content)



 -- Result 1 -- 

LangChain helps developers build LLM applications easily.

 -- Result 2 -- 

LangGraph helps developers to build Agentic system


**How vector store retriever works bahind the scene**
Flow :
1. Documents are converted into embeddings using an embedding model
2. Embeddings, original text, and metadata are stored in a vector database
3. User query is also converted into an embedding using the same model
4. Vector database (FAISS / Chroma) performs semantic similarity comparison
5. Retriever fetches the most relevant documents
6. Original text documents are returned (not vectors)

# Maximal Marginal Relevance (MMR)

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
docs_mmr = [
    Document(page_content="LangChain is a framework used to build applications with large language models."),
    Document(page_content="LangChain is a framework that helps developers create LLM-powered applications."),
    Document(page_content="LangChain provides tools and abstractions for developing applications using LLMs."),

    Document(page_content="LangGraph is a framework for building stateful and multi-step LLM workflows."),
    Document(page_content="LangGraph helps define graph-based execution flows for LLM applications."),

    Document(page_content="LangSmith is a platform for debugging, monitoring, and evaluating LLM applications."),
    Document(page_content="LangSmith helps track, test, and analyze LLM application performance."),
]

In [ ]:
embedding = GoogleGenerativeAIEmbeddings(
  model = "models/text-embedding-004"
)

In [ ]:
vectorstore = FAISS.from_documents(
    embedding = embedding,
    documents = docs_mmr
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k" : 3, "lambda_mult" : 0.5} # lambda_mult = relevance-diversity balance
)

In [ ]:
query = "What is Langchain?"

In [ ]:
results = retriever.invoke(query)

In [ ]:
for i, item in enumerate(results):
  print(f"\n -- Result -- \n {i+1} \n")
  print(f"{item.page_content}")


 -- Result -- 
 1 

LangChain is a framework used to build applications with large language models.

 -- Result -- 
 2 

LangSmith helps track, test, and analyze LLM application performance.

 -- Result -- 
 3 

LangGraph helps define graph-based execution flows for LLM applications.


**How MMR works?**
same flow like similarity search retriever/data store retriever till user_query converted to vectors and vector store FAISS/Chroma performs similarity search

1.   MMR selects the most relevant document first
2.   If a document repeats the same context, it is skipped next one gets selected
3. Next closest but contextually different document is selected
4. MMR Retriever fetches and returns the final set of documents

Note : **"lambda_mult"** has value from 0-1 so, lower value means higher diversity, higher value means lower diversity and **safe value is 0.5** where retriever didn't pick same contextual repeated documents a response